# Stack health check
Is each service of Bluecore up, and which version is it running compared with the latest release?

Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

## Health endpoints
The same health paths the dashboard's Version Drift page checks, plus Keycloak's `bluecore` realm.

In [ ]:
import time
import pandas as pd
import requests

CHECKS = {
    "Blue Core API": "/api/health",
    "Workflows (Airflow)": "/workflows/health",
    "Sinopia": "/sinopia/health",
    "Marva": "/marva/health",
    "Keycloak (bluecore realm)": "/keycloak/realms/bluecore/.well-known/openid-configuration",
}

rows = []
for name, path in CHECKS.items():
    started = time.monotonic()
    try:
        r = bc.get(path, timeout=10)
        ms = round((time.monotonic() - started) * 1000)
        try:
            body = r.json()
        except ValueError:
            body = {}
        version = body.get("version") if isinstance(body, dict) else None
        rows.append({"service": name, "status": r.status_code, "ok": r.ok, "version": version, "ms": ms})
    except requests.RequestException as err:
        rows.append({"service": name, "status": None, "ok": False, "version": None, "ms": None, "error": type(err).__name__})

health = pd.DataFrame(rows)
health

## Compared with the latest releases
From the dashboard's live Version Drift data (GitHub), so you can see whether this environment is behind.

In [ ]:
drift = requests.get(f"{DASHBOARD_API}/github/drift/blue-core-lod", timeout=60).json()
releases = pd.DataFrame([
    {
        "service": s["name"],
        "latest release": (s.get("release") or {}).get("tag"),
        "unreleased commits on main": (s.get("unreleased") or {}).get("count", 0),
        "stack image built": (s.get("imageBuiltAt") or "")[:10],
        "summary": s.get("summary"),
    }
    for s in drift["services"]
])
releases

In [ ]:
down = health[~health["ok"]]
if down.empty:
    print("Every service answered.")
else:
    print("Not answering:", ", ".join(down["service"]))
    if bc.deployed:
        print("Compare with the other environments in the next starter, or on the dashboard's Version Drift page.")
    else:
        print("Check `docker compose ps` in bluecore-stack, and the service's logs (`docker compose logs <service>`).")